In [0]:
storage_account_name = "stenergydatos"
storage_account_key = "TU_KEY_AQUI"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://silver@stenergydatos.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://silver@stenergydatos.dfs.core.windows.net/produccion/,produccion/,0,1779067752000


In [0]:
import requests
import json

In [0]:
df = spark.read.parquet(
    "abfss://silver@stenergydatos.dfs.core.windows.net/produccion/"
)

In [0]:
display(df)

Entity,Code,Year,Oil
Argentina,ARG,1900,0
Argentina,ARG,1901,0
Argentina,ARG,1902,0
Argentina,ARG,1903,0
Argentina,ARG,1904,0
Argentina,ARG,1905,0
Argentina,ARG,1906,0
Argentina,ARG,1907,0
Argentina,ARG,1908,0.02326
Argentina,ARG,1909,0.03489


In [0]:
df.printSchema()

root
 |-- Entity: string (nullable = true)
 |-- Code: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Oil: string (nullable = true)



Renombrar columnas

In [0]:
df = df.withColumnRenamed("Entity", "pais") \
       .withColumnRenamed("Code", "codigo_pais") \
       .withColumnRenamed("Year", "anio") \
       .withColumnRenamed("Oil", "produccion_petroleo")

In [0]:
display(df)

pais,codigo_pais,anio,produccion_petroleo
Argentina,ARG,1900,0
Argentina,ARG,1901,0
Argentina,ARG,1902,0
Argentina,ARG,1903,0
Argentina,ARG,1904,0
Argentina,ARG,1905,0
Argentina,ARG,1906,0
Argentina,ARG,1907,0
Argentina,ARG,1908,0.02326
Argentina,ARG,1909,0.03489


Convertir tipos de datos

In [0]:
from pyspark.sql.functions import col

df = df.withColumn(
        "anio",
        col("anio").cast("int")
     ).withColumn(
        "produccion_petroleo",
        col("produccion_petroleo").cast("double")
     )

In [0]:
df.printSchema()

root
 |-- pais: string (nullable = true)
 |-- codigo_pais: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- produccion_petroleo: double (nullable = true)



País no nulos

In [0]:
df = df.filter(col("pais").isNotNull())

Producción positiva

In [0]:
df = df.filter(col("produccion_petroleo") >= 0)

Año válido

In [0]:
df = df.filter(col("anio") >= 1900)

In [0]:
print(df.count())

1810


Estadística Descriptiva

In [0]:
display(df.describe())

summary,pais,codigo_pais,anio,produccion_petroleo
count,1810,1810,1810,1810
mean,null,null,1963.0773480662983,760.9064694756352
stddev,null,null,36.1020080727188,1410.8168792728393
min,Argentina,ARE,1900,0.0
max,Venezuela,VEN,2024,9977.384


Top productores

In [0]:
top_paises = df.groupBy("pais") \
    .sum("produccion_petroleo") \
    .orderBy("sum(produccion_petroleo)", ascending=False)

display(top_paises)

pais,sum(produccion_petroleo)
United States,447230.03297000006
Saudi Arabia,295174.58085
Venezuela,124371.49687999999
Canada,95907.81292999999
Mexico,87743.62044
Iraq,85479.14167000001
United Arab Emirates,75828.32879
Norway,52272.11675729999
Qatar,28865.7545
Argentina,23184.737309999993


In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://silver@stenergydatos.dfs.core.windows.net/produccion_limpia/"
    )

Conectando con la API 

In [0]:
import requests

api_key = "TU_KEY_AQUI"
url = "https://api.oilpriceapi.com/v1/prices/latest"

headers = {
    "Authorization": f"Token {api_key}"
}

response = requests.get(url, headers=headers)
data = response.json()
print(data)

{'status': 'success', 'data': {'price': 111.23, 'formatted': '$111.23', 'currency': 'USD', 'code': 'BRENT_CRUDE_USD', 'created_at': '2026-05-18T03:13:24.656Z', 'updated_at': '2026-05-18T03:13:24.656Z', 'type': 'spot_price', 'unit': 'barrel', 'source': 'oilprice.ft', 'data_status': 'current', 'freshness': {'status': 'current', 'age_seconds': 44, 'expected_max_age_seconds': 3600}, 'changes': {'24h': {'amount': 1.76, 'percent': 1.61, 'previous_price': 109.47}}, 'metadata': {'source': 'oilprice.ft', 'source_description': 'Financial Times'}}}


In [0]:
precio = data["data"]["price"]

print(precio)

111.23


In [0]:
api_data = [{
    "precio_petroleo": data["data"]["price"],
    "moneda": data["data"]["currency"],
    "codigo": data["data"]["code"],
    "unidad": data["data"]["unit"]
}]

In [0]:
api_df = spark.createDataFrame(api_data)

In [0]:
display(api_df)

codigo,moneda,precio_petroleo,unidad
BRENT_CRUDE_USD,USD,111.23,barrel


Agregamos el precio al dataset principal

In [0]:
from pyspark.sql.functions import lit

df_gold = df.withColumn(
    "precio_petroleo",
    lit(precio)
)

In [0]:
df_gold = df_gold.withColumn(
    "valor_estimado",
    col("produccion_petroleo") * col("precio_petroleo")
)

In [0]:
display(df_gold)

pais,codigo_pais,anio,produccion_petroleo,precio_petroleo,valor_estimado
Argentina,ARG,1900,0.0,111.23,0.0
Argentina,ARG,1901,0.0,111.23,0.0
Argentina,ARG,1902,0.0,111.23,0.0
Argentina,ARG,1903,0.0,111.23,0.0
Argentina,ARG,1904,0.0,111.23,0.0
Argentina,ARG,1905,0.0,111.23,0.0
Argentina,ARG,1906,0.0,111.23,0.0
Argentina,ARG,1907,0.0,111.23,0.0
Argentina,ARG,1908,0.02326,111.23,2.5872098
Argentina,ARG,1909,0.03489,111.23,3.8808146999999997


In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://gold@stenergydatos.dfs.core.windows.net/analitica/"
    )

In [0]:
display(
    dbutils.fs.ls(
        "abfss://gold@stenergydatos.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://gold@stenergydatos.dfs.core.windows.net/analitica/,analitica/,0,1779074621000


In [0]:
gold_df = spark.read.format("delta").load(
    "abfss://gold@stenergydatos.dfs.core.windows.net/analitica/"
)

In [0]:
display(gold_df)

pais,codigo_pais,anio,produccion_petroleo,precio_petroleo,valor_estimado
Argentina,ARG,1900,0.0,111.23,0.0
Argentina,ARG,1901,0.0,111.23,0.0
Argentina,ARG,1902,0.0,111.23,0.0
Argentina,ARG,1903,0.0,111.23,0.0
Argentina,ARG,1904,0.0,111.23,0.0
Argentina,ARG,1905,0.0,111.23,0.0
Argentina,ARG,1906,0.0,111.23,0.0
Argentina,ARG,1907,0.0,111.23,0.0
Argentina,ARG,1908,0.02326,111.23,2.5872098
Argentina,ARG,1909,0.03489,111.23,3.8808146999999997


Análisis y gráficos

Top países productores

In [0]:
top_productores = gold_df.groupBy("pais") \
    .sum("produccion_petroleo") \
    .orderBy("sum(produccion_petroleo)", ascending=False)

In [0]:
display(top_productores)

pais,sum(produccion_petroleo)
United States,447230.03297000006
Saudi Arabia,295174.58085
Venezuela,124371.49687999999
Canada,95907.81292999999
Mexico,87743.62044
Iraq,85479.14167000001
United Arab Emirates,75828.32879
Norway,52272.11675729999
Qatar,28865.7545
Argentina,23184.737309999993


Databricks visualization. Run in Databricks to view.

Producción por año

In [0]:
produccion_anual = gold_df.groupBy("anio") \
    .sum("produccion_petroleo") \
    .orderBy("anio")

In [0]:
display(produccion_anual)

anio,sum(produccion_petroleo)
1900,100.50646
1901,109.25222
1902,138.89709000000002
1903,157.0864
1904,183.09107999999998
1905,210.88679
1906,198.69854999999998
1907,261.73314999999997
1908,285.65604
1909,291.56410999999997


Databricks visualization. Run in Databricks to view.

Top valor estimado

In [0]:
top_valor = gold_df.groupBy("pais") \
    .sum("valor_estimado") \
    .orderBy("sum(valor_estimado)", ascending=False)

In [0]:
display(top_valor)

pais,sum(valor_estimado)
United States,4.974539656725309E7
Saudi Arabia,3.283226862794551E7
Venezuela,1.38338415979624E7
Canada,1.06678260322039E7
Mexico,9759722.9015412
Iraq,9507844.927954098
United Arab Emirates,8434385.0113117
Norway,5814227.546914479
Qatar,3210737.8730349992
Argentina,2578838.3309913


Databricks visualization. Run in Databricks to view.